In [1]:
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import torch
from interactive_seg_backend import (
    FeatureConfig,
    TrainingConfig,
    concat_feats,
    featurise,
    train_and_apply,
)
from interactive_seg_backend.file_handling import load_labels
from PIL import Image
from skimage.transform import resize

from dinosaw.utils import add_custom_font, do_2D_pca, get_features
from dinosaw.wrappers import MODEL_NAMES, ModelTypes, get_models

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:0'
half = False

2026-11-09 20:40:08 | I | multiscale_classical_cpu.py:  46 | N CPUS: 10
2026-11-09 20:40:08 | W | sam_onnx.py                :  19 | ONNX SAM unavailable!


/home/ronan/Documents/academic/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
enabled_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dvt_dinov2_s', 'alibi_dinov2_s_no_norm', 'dinov3_s+', 'alibi_dinov3_s+_norm_wrap_ms')
models = get_models(enabled_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")

f_cfg = FeatureConfig()

tr_cfg = TrainingConfig(feature_config=f_cfg, CRF=True, classifier='xgb', CRF_params={"label_confidence": 0.6,
    "sxy_g": [1, 1],
    "sxy_b": [30, 30],
    "s_rgb": [13, 13, 13],
    "compat_g": 10,
    "compat_b": 10,
    "n_infer": 10}, classifier_params={"class_weight": "balanced", "max_depth": 8}) 

2026-11-09 20:40:10 | I | factory.py                 : 180 | Building wrapper 'dinov2_s' on device cuda:0
2026-11-09 20:40:10 | I | factory.py                 : 158 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


2026-11-09 20:40:10 | I | wrapper.py                 :  57 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-11-09 20:40:10 | I | factory.py                 : 180 | Building wrapper 'dvt_dinov2_s' on device cuda:0
2026-11-09 20:40:10 | I | factory.py                 : 158 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)
2026-11-09 20:40:10 | I | wrapper.py                 :  57 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-11-09 20:40:10 | I | factory.py                 : 180 | Building wrapper 'alibi_dinov2_s_no_norm'

VisionTransformer doesn't satisfy Dv3StyleViT; missing: n_storage_tokens, rope_embed


['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_init_weights', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_parameters', '_pos_embed', '_r

DinoVisionTransformer doesn't satisfy TimmStyleViTGetIntermediates; missing: pos_embed, num_prefix_tokens


['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_intermediate_layers_not_chunked', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_paramet

In [ ]:
ds_folder = '../paper_figures_pdf/data/trainable_seg'

AllowedImages = Literal['anode', 'curtains', 'bimodal', 'NMC', 'steel', 'rome', 'butterflies', 'au_foil']


side_lengths: dict[AllowedImages, int] = {
    'au_foil': 737,
    'anode': 756,
    'curtains': 756,
    'rome': 756,
    'nmc_wide': 808,
    'cracks': 602,
    'butterflies': 756,
    'bimodal': 839,
    'NMC': 602,
    'steel': 602,
    'pre_kint': 1024,
}


image_fnames: list[AllowedImages] = ['bimodal',  'nmc_wide', 'cracks', 'pre_kint']
# image_fnames: list[AllowedImages] = ['curtains', 'anode', 'pre_kint']
images: list[Image.Image] = []
labels: list[np.ndarray] = []
for image_fname in image_fnames:
    img = Image.open(f"{ds_folder}/{image_fname}.png").convert("RGB")
    shortest = min(img.size)
    L = side_lengths[image_fname]
    sf = L / shortest
    new_size = (int(img.size[0] * sf), int(img.size[1] * sf))
    img = img.resize(new_size, resample=Image.BILINEAR)
    images.append(img)

    label = load_labels(f"{ds_folder}/{image_fname}_biased_labels.tiff")
    labels.append(label)

FileNotFoundError: [Errno 2] No such file or directory: '../paper_figures/data/trainable_seg/bimodal.png'

In [ ]:
features: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}
red_features: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}


for key, model in models.items():
    for img in images:

        classical = featurise(np.array(img.convert('L')), tr_cfg)

        if key == 'classical':
            features[key].append(classical)
            red_features[key].append(classical)
            continue

        feats = get_features(model, img, False)
        reduced = do_2D_pca(feats, 9, pre_norm='std', post_norm='minmax')
        reduced_hr = resize(reduced, (img.height, img.width), order=1, mode='reflect', anti_aliasing=True)

        combined_feats = concat_feats(reduced_hr, classical)
        features[key].append(combined_feats)

        red_features[key].append(reduced_hr)

2026-07-24 13:36:54 | I | multiscale_classical_cpu.py: 466 | CPU feats on (839, 1681) with `default`: weka-style features
2026-07-24 13:36:56 | I | main.py                    :  71 | Features out: (839, 1681, 58)
2026-07-24 13:36:56 | I | wrapper.py                 :  92 | Processing image, size: [1681, 839]
2026-07-24 13:36:56 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,826,1680] -> f: [1,384,59,120]
2026-07-24 13:36:57 | I | multiscale_classical_cpu.py: 466 | CPU feats on (808, 1518) with `default`: weka-style features
2026-07-24 13:36:58 | I | main.py                    :  71 | Features out: (808, 1518, 58)
2026-07-24 13:36:58 | I | wrapper.py                 :  92 | Processing image, size: [1518, 808]
2026-07-24 13:36:58 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,798,1512] -> f: [1,384,57,108]
2026-07-24 13:36:59 | I | multiscale_classical_cpu.py: 466 | CPU feats on (602, 1096) with `default`: weka-style features
2026-07-24 13:37:00 | 

In [ ]:
preds: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}

for key, model in models.items():
    for i, feat in enumerate(features[key]):
        image = np.array(images[i])
        label = labels[i]

        pred, _, _ = train_and_apply(feat, label, tr_cfg, image=image)
        preds[key].append(pred)

2026-07-24 13:37:47 | I | core.py                    : 155 | Training XGBClassifier: (1763, 67) -> ((1763,)) 


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [13:37:47] WARNING: /workspace/src/learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-07-24 13:37:47 | I | core.py                    : 170 | Applying XGBClassifier to (839, 1681, 67) features
2026-07-24 13:37:47 | I | crf.py                     :  77 | CRF: probs (839, 1681, 4), img (839, 1681, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}
2026-07-24 13:37:49 | I | core.py                    : 155 | Training XGBClassifier: (5764, 67) -> ((5764,)) 
2026-07-24 13:37:49 | I | core.py                    : 170 | Applying XGBClassifier to (808, 1518, 67) features
2026-07-24 13:37:49 | I | crf.py                     :  77 | CRF: probs (808, 1518, 3), img (808, 1518, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [13:37:49] WARNING: /workspace/src/learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-07-24 13:37:51 | I | core.py                    : 155 | Training XGBClassifier: (2385, 67) -> ((2385,)) 
2026-07-24 13:37:51 | I | core.py                    : 170 | Applying XGBClassifier to (602, 1096, 67) features
2026-07-24 13:37:51 | I | crf.py                     :  77 | CRF: probs (602, 1096, 3), img (602, 1096, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [13:37:51] WARNING: /workspace/src/learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


... (31 outputs left out)

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [13:38:22] WARNING: /workspace/src/learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-07-24 13:38:23 | I | core.py                    : 155 | Training XGBClassifier: (16701, 67) -> ((16701,)) 


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [13:38:23] WARNING: /workspace/src/learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-07-24 13:38:23 | I | core.py                    : 170 | Applying XGBClassifier to (1024, 1995, 67) features
2026-07-24 13:38:23 | I | crf.py                     :  77 | CRF: probs (1024, 1995, 4), img (1024, 1995, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}


In [ ]:
from PIL.ImageColor import getcolor
from skimage.color import label2rgb

COLOURS = [
    "#648FFF",
    "#785EF0",
    "#DC267F",
    "#FE6100",
    "#FFB000"
]
COLORS = [[v / 255.0 for v in getcolor(c, "RGB")] for c in COLOURS]



# add_custom_font('resources/fonts', 'Grotesk')

def hide_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])



def apply_labels_as_overlay(labels: np.ndarray, img: Image.Image, colors: list, alpha: float=1.0) -> Image.Image:
    labels_unsqueezed = np.expand_dims(labels, -1)

    overlay = label2rgb(labels, colors=colors[1:], kind='overlay', bg_label=0, image_alpha=1, alpha=alpha)
    out = np.where(labels_unsqueezed, overlay * 255, np.array(img)).astype(np.uint8)
    img_with_labels = Image.fromarray(out)
    return img_with_labels

In [ ]:
# %%capture
plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')
# W, H = 7, 2.3 * 3.25
W, H = 7.5, 2.3 * 4

N_ROWS = len(enabled_models) + 1
N_COLS = len(image_fnames)
fig, axs = plt.subplots(N_ROWS, N_COLS, figsize=(W, H))

# titles = ["Bimodal cathode", "Anode", "Cathode (+CC)"]
titles = ["Bimodal cathode", "Cathode", "Cathode (+cracks)", "Cathode (+CC)"]

for y in range(N_ROWS):
    for x in range(N_COLS):
        if y == 0:
            axs[y, 0].set_ylabel("Image + labels")
            axs[y, x].set_title(titles[x])
            arr = apply_labels_as_overlay(labels[x], images[x], COLORS, alpha=1)
        else:
            key = enabled_models[y-1]
            # model_name = MODEL_NAMES[key]
            model_name = key
            model_name = model_name.replace('(COCO)', '')
            weight = 700 if 'alibi' in key.lower() else 500
            axs[y, 0].set_ylabel(model_name, weight=weight)
            arr = preds[key][x]
            arr = label2rgb(arr + 1, colors=COLORS[1:], bg_label=0)
        axs[y, x].imshow(arr, aspect='auto', interpolation='nearest')
        hide_axis(axs[y, x])

SAVE = False
if SAVE:
    plt.savefig('saved/S15.pdf', dpi=300, bbox_inches='tight')
    plt.close()


findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight 500, now using 300.
